In [1]:
import pandas as pd
import re

# -------------------------- 1. 配置参数：领域-关键词映射（可根据需求增删） --------------------------
# 键：分类名称，值：匹配关键词（包含书名/简介中的核心词）
FIELD_KEYWORDS = {
    "Python基础入门": ["入门", "从入门到实践", "从入门到精通", "笨办法", "你好!Python", "完全自学教程"],
    "机器学习与深度学习": ["机器学习", "深度学习", "神经网络", "scikit-learn", "AI+", "自然语言处理", "强化学习"],
    "数据分析与可视化": ["数据分析", "数据科学", "数据可视化", "Pandas", "NumPy", "Matplotlib", "数亦有道"],
    "办公自动化": ["办公自动化", "Python+Office", "ChatGPT办公", "玩转Excel", "高效办公"],
    "量化交易/金融分析": ["量化交易", "金融量化", "区块链量化", "vn.py", "交易策略", "量化投资"],
    "网络爬虫": ["网络爬虫", "爬虫", "Urllib", "BeautifulSoup", "Scrapy"],
    "青少年/趣味编程": ["小学生", "青少年", "趣味编程", "看漫画学Python", "创意编程"],
    "高阶开发与配套基础": ["全栈开发", "Web编程", "现代Python编程", "矩阵", "线性代数", "程序是怎样跑起来", "智能优化算法"]
}

# -------------------------- 2. 加载CSV数据（核心修改处） --------------------------
# 替换为你的CSV文件路径（已改为目标文件名）
csv_file = "dangdang_python_books_clean.csv"
try:
    # 优先尝试utf-8编码（通用编码），失败则用gbk（中文CSV常见编码）
    df = pd.read_csv(csv_file, encoding="utf-8")
except UnicodeDecodeError:
    df = pd.read_csv(csv_file, encoding="gbk")

# 填充空值（避免匹配时报错）
df["书名"] = df["书名"].fillna("")
df["简介"] = df["简介"].fillna("")
# 合并书名+简介为匹配文本
df["match_text"] = df["书名"] + " " + df["简介"]
# 统一为小写（取消大小写影响）
df["match_text"] = df["match_text"].str.lower()

# -------------------------- 3. 定义分类匹配函数 --------------------------
def classify_book(text):
    """根据文本匹配返回所属分类，无匹配则返回「其他/跨领域融合」"""
    text = text.lower()
    for field, keywords in FIELD_KEYWORDS.items():
        # 遍历关键词，只要匹配一个即归为该分类
        for kw in keywords:
            if re.search(kw.lower(), text):
                return field
    return "其他/跨领域融合"

# 为每本书添加分类标签
df["所属分类"] = df["match_text"].apply(classify_book)

# -------------------------- 4. 归总分类信息 --------------------------
# 按分类分组，统计每类图书数量+提取核心信息（书名/作者/折后价）
classify_summary = df.groupby("所属分类").agg(
    图书数量=("书名", "count"),
    图书列表=("书名", list),
    代表作者=("作者", lambda x: list(x.dropna().unique())[:3]),  # 取前3个代表作者
    均价=("折后价", lambda x: round(x.dropna().mean(), 2))  # 折后价均价（保留2位小数）
).reset_index()

# -------------------------- 5. 输出结果 --------------------------
print("="*80)
print("Python图书分类归总结果")
print("="*80)
for _, row in classify_summary.iterrows():
    print(f"\n【{row['所属分类']}】")
    print(f"数量：{row['图书数量']}本 | 均价：{row['均价']}元 | 代表作者：{row['代表作者']}")
    print(f"图书列表：{row['图书列表']}")

# -------------------------- 6. 可选：将分类结果保存为新Excel --------------------------
df.to_excel("python_books_classified.xlsx", sheet_name="分类后数据", index=False)
classify_summary.to_excel("python_books_classify_summary.xlsx", sheet_name="分类归总", index=False)
print("\n" + "="*80)
print("分类结果已保存为：python_books_classified.xlsx、python_books_classify_summary.xlsx")

Python图书分类归总结果

【Python基础入门】
数量：29本 | 均价：83.27元 | 代表作者：['[美]埃里克・马瑟斯（EricMatthes）', '马国俊', '明日科技编著']
图书列表：['Python编程从入门到实践 第3版 Python编程入门经典，自学利器，数据分析、网络爬虫、机器学习、深度学习基础，基于Python 3.11升级，附赠配套视频、源代码、PPT课件、速查地图、学习路线图等资源', 'Python网络爬虫与数据分析从入门到实践', 'Python完全自学教程', 'Python编程:从入门到实践(第3版)', '深度学习入门 基于Python的理论与实现 人工智能入门学习教程 日本深度学习入门书籍 基于python 3 从零创建一个深度学习模型', 'Python区块链量化交易 从区块链基础知识到交易所API，再到实战交易策略 一本书带你走进区块链交易的世界，开启你的量化交易之旅', '笨办法学Python 原书第5版 通过习题学习python语言编程', '你好!Python', '小学生Python创意编程 视频教学版 本书按照入门开发者的思维方式编写，非常适合孩子学习Python编程的基础知识。语言风趣幽默，讲解细致入微，案例生动有趣，能够让小朋友轻松愉悦地学习Python编程。', '奇思妙想：Python青少年趣味编程101例（视频教学版） 读故事讲知识学编程，场景化教学，培养编程兴趣，锤炼编程思维，趣味教学快乐学编程', 'Python从入门到精通 第3版 Python入门经典，26万Python程序员的入行选择。配备升级版Python开发资源库，在线大咖课+在线答疑，学习1小时，训练10小时，从入门到项目上线，打造全新学习生态。', 'Python网络爬虫入门到实战', 'Python编程：从入门到实践', 'Python编程三剑客第3版：Python编程从入门到实践第3版+快速上手第2版+极客项目编程第2版（当当套装共3册） Python编程入门经典，自学利器，数据分析、网络爬虫、机器学习、深度学习基础，基于Python 3.11升级，附赠配套视频、源代码、PPT课件、速查地图、学习路线图等资源', '智能优化算法改进：从入门到MATLAB、Python编程实践 啃不动算法改进？经典案例拆解+双语言代码+避